# Project 2

## Configuration

In [ ]:
import time
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

In [ ]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [ ]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.show(5)

## Bronze layer

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze_trips (
    key STRING,
    value STRING,
    topic STRING,
    partition INT,
    offset BIGINT,
    timestamp TIMESTAMP
)
USING iceberg
""")

In [ ]:
# import shutil

# # Clean up any leftover files from a previous run of this cell
# shutil.rmtree("/tmp/bronze",     ignore_errors=True)
# shutil.rmtree("/tmp/chk-bronze", ignore_errors=True)

bronze_source = (
    raw_stream
    .select(
        F.col("key").cast("string"),
        F.col("value").cast("string"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp")
    )
)

bronze_query = (
    bronze_source.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .toTable("lakehouse.taxi.bronze_trips")
)

time.sleep(12)

count_before = spark.read.table("lakehouse.taxi.bronze_trips").count()
print(f"Row count before restart: {count_before}")

bronze_query.stop()
print("Query stopped.")

# Restart the query using the SAME checkpoint directory.
# Even though startingOffsets is still 'earliest', Spark ignores that setting
# and resumes from the committed offsets in the checkpoint.

bronze_query2 = (
    bronze_source.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .toTable("lakehouse.taxi.bronze_trips")
)

time.sleep(12)   # wait for two triggers

count_after = spark.read.table("lakehouse.taxi.bronze_trips").count()
print(f"Row count before restart : {count_before}")
print(f"Row count after restart  : {count_after}")
print()
if count_after == count_before:
    print("Counts are equal — the checkpoint prevented re-ingestion of already-processed messages.")
else:
    print(f"Counts differ by {count_after - count_before} — "
          "those are new messages produced between the two runs.")

bronze_query2.stop()

In [ ]:
spark.sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT topic, partition, offset) AS unique_events
FROM lakehouse.taxi.bronze_trips;""").show()

## Silver layer

In [ ]:
# Load bronze data
bronze_df = spark.read.table("lakehouse.taxi.bronze_trips")

# Load zone lookup
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")

# inspect
print("Bronze count:", bronze_df.count())
zones.show(5)
zones.printSchema()

In [ ]:
zones.cache()
zones.count()  # materialize cache

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, TimestampType

# Define schema of your taxi data
schema = StructType([
    StructField("VendorID", IntegerType()),
    StructField("tpep_pickup_datetime", TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count", IntegerType()),
    StructField("trip_distance", DoubleType()),
    StructField("PULocationID", IntegerType()),
    StructField("DOLocationID", IntegerType()),
    StructField("fare_amount", DoubleType()),
    StructField("total_amount", DoubleType())
])

# Parse JSON from Kafka "value"
parsed_df = (
    bronze_df
    .withColumn("json", F.from_json(F.col("value").cast("string"), schema))
    .select("json.*")
)

In [ ]:
parsed_df.select("*").show(5, truncate=False)
print("Parsed count:", parsed_df.count())

In [ ]:
bronze_df.selectExpr("CAST(value AS STRING)").show(5, False)

In [ ]:
silver_df = parsed_df.select(
    F.col("VendorID").cast("int"),

    F.to_timestamp("tpep_pickup_datetime").alias("tpep_pickup_datetime"),
    F.to_timestamp("tpep_dropoff_datetime").alias("tpep_dropoff_datetime"),

    # FIX: handle null/zero passengers
    F.when(
        (F.col("passenger_count").isNull()) | (F.col("passenger_count") <= 0),
        1
    ).otherwise(F.col("passenger_count")).cast("int").alias("passenger_count"),

    F.col("trip_distance").cast("double"),
    F.col("PULocationID").cast("int"),
    F.col("DOLocationID").cast("int"),
    F.col("fare_amount").cast("double"),
    F.col("total_amount").cast("double")
)

In [ ]:
silver_df = (
    silver_df

    # Drop critical nulls
    .dropna(subset=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID"
    ])

    # Valid trip duration
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))

    # Positive distance
    .filter(F.col("trip_distance") > 0)

    # Passenger sanity
    .filter((F.col("passenger_count") > 0) & (F.col("passenger_count") < 10))

    # Non-negative fare
    .filter(F.col("fare_amount") >= 0)

    # Remove duplicates
    .dropDuplicates()
)

In [ ]:
pu = zones.alias("pu")
do = zones.alias("do")

silver_enriched_df = (
    silver_df
    .join(pu, silver_df.PULocationID == F.col("pu.LocationID"), "left")
    .join(do, silver_df.DOLocationID == F.col("do.LocationID"), "left")
    .select(
        silver_df["*"],

        F.col("pu.Borough").alias("pickup_borough"),
        F.col("pu.Zone").alias("pickup_zone"),
        F.col("pu.service_zone").alias("pickup_service_zone"),

        F.col("do.Borough").alias("dropoff_borough"),
        F.col("do.Zone").alias("dropoff_zone"),
        F.col("do.service_zone").alias("dropoff_service_zone")
    )
)

In [ ]:
silver_enriched_df = silver_enriched_df.filter(
    F.col("pickup_zone").isNotNull() &
    F.col("dropoff_zone").isNotNull()
)

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.silver_trips (
    VendorID INT,
    tpep_pickup_datetime TIMESTAMP,
    tpep_dropoff_datetime TIMESTAMP,
    passenger_count INT,
    trip_distance DOUBLE,
    PULocationID INT,
    DOLocationID INT,
    fare_amount DOUBLE,
    total_amount DOUBLE,
    pickup_borough STRING,
    pickup_zone STRING,
    pickup_service_zone STRING,
    dropoff_borough STRING,
    dropoff_zone STRING,
    dropoff_service_zone STRING
)
USING ICEBERG
""")

In [ ]:
(
    silver_enriched_df
    .writeTo("lakehouse.taxi.silver_trips")
    .append()
)

In [ ]:
silver_check = spark.read.table("lakehouse.taxi.silver_trips")

silver_check.show(10, truncate=False)
print("Silver count:", silver_check.count())

In [ ]:
silver_check.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_check.columns
]).show()

In [ ]:
silver_check.groupBy("pickup_borough").count().show()

In [ ]:
silver_check.select(
    (F.col("tpep_dropoff_datetime").cast("long") - 
     F.col("tpep_pickup_datetime").cast("long")).alias("duration_sec")
).describe().show()

stored in: "lakehouse.taxi.silver_trips" 